# Asimov closure (step2, flavor)
Flavor triangle scans on an Asimov dataset injected at the HESE best fit with a 1:1:1 flavor composition
(`dag_scans/.../step2_hese_flavor_reunblind/asimov_closure`, see `asimov_closure.sh`).
The best fit should recover the injected parameters, and the contours show the expected sensitivity.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
import yaml
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from NNMFit.utilities import ScanHandler

# import plotting utils
sys.path.append( "/data/user/tvaneede/utils" )
from FlavourScansPlotting import get_contour_points

# import plotting flavor triangle
sys.path.append( "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks" )
from plot_utils_triangle import *

In [ ]:
plotting_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor_reunblind/studies/plots_asimov_closure"
os.makedirs(plotting_path, exist_ok=True)

In [ ]:
dag_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor_reunblind/asimov_closure"
# _muonconv: old muon template (muontemplate_hese, muongun_norm) of the paper instead of the combined conv/prompt template
# _muonconvneha: same single muon template, but the non-paper KDE (HESE_Tracks_KDE_..._bdt2_0.366667.pickle)
# _LBFGSB: plain LBFGSB minimizer instead of LBFGSB_force (3 full fits, 20 light scans)
# _oldsim: old simulation and data sets (Combined_v7-v1, data_HESE_pass2) instead of v8-v2_PassingHESE / pass2_v5
# _oldnnm: NNMFit v0.3.0 venv instead of NNMFit_dev-new-flavor-param-2
# _oldlive: no hese_livetime_13yr_asr.cfg override (default livetime)
scan_dir_dict = {
    "SAY"                     : f"{dag_path}/SAY",
    "SAY_nosyst"              : f"{dag_path}/SAY_nosyst",
    "SAY_nosyst_muonconv"     : f"{dag_path}/SAY_nosyst_muonconv",
    "SAY_nosyst_muonconvneha"               : f"{dag_path}/SAY_nosyst_muonconvneha",
    "SAY_nosyst_muonconvneha_LBFGSB"        : f"{dag_path}/SAY_nosyst_muonconvneha_LBFGSB",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim" : f"{dag_path}/SAY_nosyst_muonconvneha_LBFGSB_oldsim",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm"         : f"{dag_path}/SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive" : f"{dag_path}/SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive",
    "Poisson"                 : f"{dag_path}/Poisson",
    "Poisson_nosyst"          : f"{dag_path}/Poisson_nosyst",
    "Poisson_nosyst_muonconv" : f"{dag_path}/Poisson_nosyst_muonconv",
}
labels = {
    "SAY"                     : "SAY",
    "SAY_nosyst"              : "SAY, no systematics",
    "SAY_nosyst_muonconv"     : "SAY, no syst., paper muon template",
    "SAY_nosyst_muonconvneha"               : "SAY, no syst., KDE muon template",
    "SAY_nosyst_muonconvneha_LBFGSB"        : "SAY, no syst., KDE muon, LBFGSB",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim" : "SAY, no syst., KDE muon, LBFGSB, old sim.",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm"         : "SAY, no syst., KDE muon, LBFGSB, old sim., NNMFit v0.3.0",
    "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive" : "SAY, no syst., KDE muon, LBFGSB, old sim., NNMFit v0.3.0, default livetime",
    "Poisson"                 : "Poisson",
    "Poisson_nosyst"          : "Poisson, no systematics",
    "Poisson_nosyst_muonconv" : "Poisson, no syst., paper muon template",
}

In [ ]:
N_grid = 40
drop_unsuccessful = False
scan_dict = {}
data = {}
bestfits = {}
for name, scan_dir in scan_dir_dict.items():
    print(name)
    data[name] = {}
    scan_dict[name] = ScanHandler(scan_dir,dump=False,drop_unsuccessful=drop_unsuccessful)
    data[name]["ftau"],data[name]["fe"],data[name]["ts"],data[name]["ftau_grid"],data[name]["fe_grid"],data[name]["ts_grid"] = get_contour_points(scan_dict,name, N_grid=N_grid)
    bestfits[name] = scan_dict[name].get_freefit()

n_points = len(pd.read_pickle("/data/user/tvaneede/GlobalFit/custom_scan_flavor/default_custom_scan_points_flavor.pickle")["astro_nue_ratio"])
# get_contour_points only uses successful fits
for name, scan_dir in scan_dir_dict.items():
    n_files = len([f for f in os.listdir(scan_dir) if f.startswith("FitRes")])
    print(f"{name}: {n_files} / {n_points} scan points finished, {len(data[name]['ts'])} successful, freefit llh = {bestfits[name]['llh']:.4f}")

## Injected parameters
Injected values are the `input_params` of the asimov analysis config, all other parameters are at their default.

In [ ]:
def get_injected(scan_dir):
    with open(f"{scan_dir}/Fit_Configuration.yaml") as f:
        config = yaml.safe_load(f)

    injected = {}
    for component in config["components"].values():
        for parameter, spec in (component.get("parameters") or {}).items():
            if isinstance(spec, dict) and "default" in spec:
                injected[parameter] = float(spec["default"])

    # SnowStorm gradients (nested somewhere in the config): parameters and defaults are comma separated lists
    def find_snowstorm(block):
        if not isinstance(block, dict):
            return
        if block.get("class") == "SnowStormGradient":
            names = [p.strip() for p in block["parameters"].split(",")]
            defaults = [float(v) for v in str(block["default"]).split(",")]
            injected.update(dict(zip(names, defaults)))
            return
        for sub in block.values():
            find_snowstorm(sub)
    find_snowstorm(config.get("config", {}))

    injected.update({p: float(v) for p, v in config["analysis"]["input_params"].items()})
    return injected

def flavor_fractions(params):
    norm = 1 + params["astro_nue_ratio"] + params["astro_nutau_ratio"]
    return {"f_e": params["astro_nue_ratio"] / norm, "f_mu": 1 / norm, "f_tau": params["astro_nutau_ratio"] / norm}

injected = {name: get_injected(scan_dir) for name, scan_dir in scan_dir_dict.items()}

## Best fit vs injected

In [ ]:
signal_parameters = ["astro_norm", "gamma_astro", "astro_nue_ratio", "astro_nutau_ratio"]

closure = {}
for name in scan_dir_dict:
    bf = bestfits[name]
    rows = {}
    for parameter in signal_parameters + sorted(p for p in injected[name] if p not in signal_parameters):
        if parameter not in bf:
            continue  # parameter not fitted (e.g. fixed or not present in this scan)
        rows[parameter] = {"injected": injected[name][parameter], "bestfit": bf[parameter]}
    for frac, value in flavor_fractions(injected[name]).items():
        rows[frac] = {"injected": value, "bestfit": flavor_fractions(bf)[frac]}
    df = pd.DataFrame(rows).T
    df["bestfit - injected"] = df["bestfit"] - df["injected"]
    closure[name] = df

for name, df in closure.items():
    print(f"=== {labels[name]}  (llh = {bestfits[name]['llh']:.4f})")
    with pd.option_context("display.float_format", "{:.4f}".format):
        display(df)

In [ ]:
# signal parameters side by side
df_signal = pd.concat({labels[name]: closure[name].loc[signal_parameters + ["f_e", "f_mu", "f_tau"], ["bestfit"]]["bestfit"]
                       for name in scan_dir_dict}, axis=1)
df_signal.insert(0, "injected", closure["SAY"].loc[df_signal.index, "injected"])
with pd.option_context("display.float_format", "{:.4f}".format):
    display(df_signal)

## Flavor triangles

In [ ]:
levels = ["68%", "90%", "95%"]
ts_values = [ts_dict[l] for l in levels]

def scatter_point(tax, params, **kwargs):
    fr = flavor_fractions(params)
    kwargs.setdefault("zorder", 10)
    return tax.ca.scatter([fr["f_tau"]], [fr["f_e"]], edgecolor="k", lw=0.5, **kwargs)

def composition(params):
    fr = flavor_fractions(params)
    return f"{fr['f_e']:.3f} : {fr['f_mu']:.3f} : {fr['f_tau']:.3f}"

def plot_closure_triangle(data, name, injected, bestfit, savepath=None):
    fig = plt.figure(figsize=(7,6))
    tax = flavor_triangle()

    C = tax.ca.contour(data[name]["ftau_grid"], data[name]["fe_grid"], data[name]["ts_grid"],
                       ts_values, linestyles=linestyles[:len(levels)], linewidths=1.5, colors=colours[0])

    lh, ll = [], []
    lh.append(scatter_point(tax, injected, marker="*", facecolor="white", s=180, zorder=11))
    ll.append(f"Injected: {composition(injected)}")
    lh.append(scatter_point(tax, bestfit, marker="X", facecolor="red", s=90))
    ll.append(f"Best fit: {composition(bestfit)}")
    for ls, level in zip(linestyles, levels):
        lh.append(Line2D([], [], color=colours[0], linestyle=ls))
        ll.append(level)
    fig.legend(lh, ll, bbox_to_anchor=(0.7, 0.05), prop=font_legend)

    plt.title(labels[name], y=1.08, fontdict=font_title)
    if savepath:
        plt.savefig(savepath, bbox_inches="tight")
    plt.show()

for name in scan_dir_dict:
    plot_closure_triangle(data, name, injected[name], bestfits[name],
                          savepath=f"{plotting_path}/asimov_closure_{name}.png")

In [ ]:
def plot_compare(names, savepath):
    fig = plt.figure(figsize=(7,6))
    tax = flavor_triangle()

    lh, ll = [], []
    lh.append(scatter_point(tax, injected[names[0]], marker="*", facecolor="white", s=180, zorder=11))
    ll.append(f"Injected: {composition(injected[names[0]])}")

    for i, name in enumerate(names):
        tax.ca.contour(data[name]["ftau_grid"], data[name]["fe_grid"], data[name]["ts_grid"],
                       ts_values, linestyles=linestyles[:len(levels)], linewidths=1.5, colors=colours[i])
        scatter_point(tax, bestfits[name], marker="X", facecolor=colours[i], s=90)
        lh.append(Line2D([], [], color=colours[i], marker="X", markeredgecolor="k", markersize=8))
        ll.append(f"{labels[name]} (best fit {composition(bestfits[name])})")

    for ls, level in zip(linestyles, levels):
        lh.append(Line2D([], [], color="grey", linestyle=ls))
        ll.append(level)
    fig.legend(lh, ll, bbox_to_anchor=(0.8, 0.05), prop=font_legend)

    plt.savefig(savepath, bbox_inches="tight")
    plt.show()

compare_groups = {
    "compare":         ["SAY", "SAY_nosyst", "Poisson"],
    "compare_SAY":     ["SAY", "SAY_nosyst", "SAY_nosyst_muonconv"],
    "compare_Poisson": ["Poisson", "Poisson_nosyst", "Poisson_nosyst_muonconv"],
    "compare_nosyst":  ["SAY_nosyst", "Poisson_nosyst", "SAY_nosyst_muonconv", "Poisson_nosyst_muonconv"],
    # SAY without systematics: muon template, then minimizer and simulation
    "compare_SAY_muon_template": ["SAY_nosyst", "SAY_nosyst_muonconv", "SAY_nosyst_muonconvneha"],
    "compare_SAY_minimizer_sim": ["SAY_nosyst_muonconvneha", "SAY_nosyst_muonconvneha_LBFGSB", "SAY_nosyst_muonconvneha_LBFGSB_oldsim"],
    "compare_SAY_nnmfit_livetime": ["SAY_nosyst_muonconvneha_LBFGSB_oldsim", "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm",
                                    "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive"],
}
for group, names in compare_groups.items():
    plot_compare(names, f"{plotting_path}/asimov_closure_{group}.png")


## Comparison with the earlier benchmark scan
`SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive` should reproduce the earlier Asimov benchmark
`dag_scans/flavor_globalfit/hese/SAY/hese_HESEBestfit_SPL_noinel_nosyst` (from `dag_scans/flavor_globalfit/create_dag_hese.sh`).
Both use the same command: `asimov/SAY/hese/hese_HESEBestfit_SPL.yaml`, `main.cfg`, the NNMFit v0.3.0 venv,
`NoSystematics_hese.cfg`, the old `..._length_10_fluxlessweight.cfg` datasets, the same binning, no livetime override,
`astro_SPL_no_inel.yaml`, the non-paper KDE muon template and the same custom scan points; only `n_bestfit 10` and `scan_memory` differ
(the reference has a single freefit), both use the plain LBFGSB minimizer.

Differences in `Fit_Configuration.yaml` (`main.cfg` changed in the meantime), none of which should affect this fit:
- the reference still defines the `inel_scale` parameter, unused with `astro_SPL_no_inel.yaml`;
- the reference has an empty `data` path (not used for an Asimov fit);
- the muon norm of the cascade globalfit template is renamed and the prior width of the track globalfit `muon_norm` differs,
  both globalfit components are skipped for the HESE detector configs;
- different SnowStorm gradient blocks are defined, both fits run with `NoSystematics_hese.cfg`.

In [ ]:
reference_name = "reference_noinel_nosyst"
reference_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/SAY/hese_HESEBestfit_SPL_noinel_nosyst"
new_name = "SAY_nosyst_muonconvneha_LBFGSB_oldsim_oldnnm_oldlive"

labels[reference_name] = "earlier benchmark"
data[reference_name] = {}
scan_dict[reference_name] = ScanHandler(reference_dir, dump=False, drop_unsuccessful=drop_unsuccessful)
data[reference_name]["ftau"],data[reference_name]["fe"],data[reference_name]["ts"],data[reference_name]["ftau_grid"],data[reference_name]["fe_grid"],data[reference_name]["ts_grid"] = get_contour_points(scan_dict, reference_name, N_grid=N_grid)
bestfits[reference_name] = scan_dict[reference_name].get_freefit()
injected[reference_name] = get_injected(reference_dir)

for name in [reference_name, new_name]:
    print(f"{labels[name]}: freefit llh = {bestfits[name]['llh']:.4f}, {len(data[name]['ts'])} successful scan points")

In [ ]:
# best fit of both scans for all parameters they have in common
rows = {}
for parameter in signal_parameters + sorted(p for p in injected[new_name] if p not in signal_parameters):
    if parameter in bestfits[reference_name] and parameter in bestfits[new_name]:
        rows[parameter] = {"injected": injected[new_name][parameter],
                           "reference": bestfits[reference_name][parameter], "new": bestfits[new_name][parameter]}
for frac in ["f_e", "f_mu", "f_tau"]:
    rows[frac] = {"injected": flavor_fractions(injected[new_name])[frac],
                  "reference": flavor_fractions(bestfits[reference_name])[frac], "new": flavor_fractions(bestfits[new_name])[frac]}
rows["llh"] = {"injected": np.nan, "reference": bestfits[reference_name]["llh"], "new": bestfits[new_name]["llh"]}
df_reference = pd.DataFrame(rows).T
df_reference["new - reference"] = df_reference["new"] - df_reference["reference"]
with pd.option_context("display.float_format", "{:.5f}".format):
    display(df_reference)

In [ ]:
# profile point by point: successful scan points present in both scans
def scan_points(name):
    return pd.DataFrame({"ftau": np.round(data[name]["ftau"], 6), "fe": np.round(data[name]["fe"], 6), "ts": data[name]["ts"]})

common = scan_points(reference_name).merge(scan_points(new_name), on=["ftau", "fe"], suffixes=("_reference", "_new"))
delta = common["ts_new"] - common["ts_reference"]
print(f"common successful scan points: {len(common)}")
print(f"TS(new) - TS(reference): max |diff| = {np.abs(delta).max():.4g}, mean |diff| = {np.abs(delta).mean():.4g}, mean diff = {delta.mean():+.4g}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(common["ts_reference"], common["ts_new"], s=15, color="C0")
lim = [0, max(common["ts_reference"].max(), common["ts_new"].max()) * 1.05]
ax.plot(lim, lim, color="black", linestyle="--", linewidth=1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel(r"$-2\Delta\log\mathcal{L}$ earlier benchmark", fontdict=font_axis_label)
ax.set_ylabel(r"$-2\Delta\log\mathcal{L}$ " + "_oldlive", fontdict=font_axis_label)
plt.savefig(f"{plotting_path}/asimov_closure_reference_ts_scatter.png", bbox_inches="tight")
plt.show()

In [ ]:
plot_compare([reference_name, new_name], f"{plotting_path}/asimov_closure_compare_reference.png")

**Outcome:** `_oldlive` reproduces the earlier benchmark. The best fits agree to better than 0.002 in every parameter
(astro_norm 2.1238 vs 2.1242, muongun_norm 0.9787 vs 0.9772, identical 0.333 : 0.333 : 0.334 composition) and the freefit llh to 0.0003
(134.3638 vs 134.3636). On the 60 scan points that converged in both scans the profile agrees within |ΔTS| ≤ 0.16 (mean 0.03,
the new scan is systematically slightly higher, i.e. the scan-point minimizations differ a little). The small differences in the
contours come mostly from the different sets of converged scan points (79 vs 82 of 103) entering the interpolation.